# AMS-02 Proton & Helium Walkthrough

End-to-end demonstration of the `ams02wb` pipeline applied to both proton
and helium cosmic-ray flux data from AMS-02.  The notebook walks through:

1. CSV ingestion via `parsers.csv_parser`
2. Provenance inspection
3. Building `Measurement` objects from parsed records
4. Schema validation of loaded records
5. Harmonisation with explicit uncertainty labelling
6. Diagonal likelihood / covariance construction
7. Fit-ready dataset assembly
8. Parquet export and round-trip verification

Both species are processed in parallel so the reader sees the full
multi-species workflow, not just a single-species toy example.

## 1. Imports

Load modules from the `ams02wb` package: parsers, schema, harmoniser,
likelihood, and exports.

In [ ]:
import io
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from ams02wb.parsers.csv_parser import parse_csv
from ams02wb.parsers.context import ParseContext
from ams02wb.schema.models import Measurement, UncertaintyLabel, ProvenanceRecord
from ams02wb.schema.validators import validate_energy_fields
from ams02wb.harmoniser.pipeline import run_harmonisation_pipeline
from ams02wb.likelihood.diagonal import build_diagonal_covariance, diagonal_chi2, UNCERTAINTY_LABEL, MODE
from ams02wb.likelihood.fitready import build_fit_dataset
from ams02wb.exports.parquet import export_parquet

## 2. Load sample CSVs

Read both proton and helium sample CSVs using the `parse_csv` interface.
Each CSV has columns: `x_min`, `x_max`, `y_value`, `stat_err`,
`sys_err_total`, `provenance_json`.

In [ ]:
proton_path = Path("../data/samples/proton_sample.csv")
helium_path = Path("../data/samples/helium_sample.csv")

with open(proton_path, encoding="utf-8") as fh:
    proton_records = parse_csv(fh)

with open(helium_path, encoding="utf-8") as fh:
    helium_records = parse_csv(fh)

print(f"Parsed {len(proton_records)} proton rows")
print(f"Parsed {len(helium_records)} helium rows")
print("\nProton first 3 rows:")
for r in proton_records[:3]:
    print(r)
print("\nHelium first 3 rows:")
for r in helium_records[:3]:
    print(r)

## 3. Inspect provenance_json

Each row carries a `provenance_json` field that traces the data back to
its source paper and table.  This is critical for reproducibility.

In [ ]:
print("=== Proton provenance ===")
for row in proton_records[:3]:
    prov = row["provenance_json"]
    print(f"  provenance_json: {prov}")

print("\n=== Helium provenance ===")
for row in helium_records[:3]:
    prov = row["provenance_json"]
    print(f"  provenance_json: {prov}")

## 4. Build Measurement objects

Convert the flat dicts from `parse_csv` to `Measurement` instances.
Asymmetric errors are set equal (symmetric) since the sample CSVs
provide a single `stat_err` and `sys_err_total` per bin.

In [ ]:
def records_to_measurements(records, species):
    """Convert parse_csv output to Measurement list."""
    measurements = []
    for r in records:
        m = Measurement(
            energy_low=r["x_min"],
            energy_high=r["x_max"],
            energy_mid=(r["x_min"] + r["x_max"]) / 2.0,
            value=r["y_value"],
            species=species,
            stat_err_pos=r["stat_err"],
            stat_err_neg=r["stat_err"],
            sys_err_pos=r["sys_err_total"],
            sys_err_neg=r["sys_err_total"],
        )
        measurements.append(m)
    return measurements

proton_measurements = records_to_measurements(proton_records, species="PROTON")
helium_measurements = records_to_measurements(helium_records, species="HELIUM")

print(f"{len(proton_measurements)} proton Measurement objects")
print(f"{len(helium_measurements)} helium Measurement objects")
print("\nFirst proton measurement:")
proton_measurements[0]

## 5. Validate loaded records

Before harmonisation, run schema validation on the loaded records.
`validate_record` checks physical bounds on energy fields
(0.1– 5000 GeV), flux positivity, and statistical-error sign.
An empty findings list means the record passed all checks.

In [ ]:
from ams02wb.schema.validators import validate_record

def measurement_to_validator_dict(m):
    """Map a Measurement to the dict format expected by validate_record."""
    return {
        "energy_centre_gev": m.energy_mid,
        "energy_low_gev": m.energy_low,
        "energy_high_gev": m.energy_high,
        "flux": m.value,
        "flux_err_stat_lo": m.stat_err_neg if m.stat_err_neg is not None else 0.0,
        "flux_err_stat_hi": m.stat_err_pos if m.stat_err_pos is not None else 0.0,
    }

proton_findings = []
for m in proton_measurements:
    findings = validate_record(measurement_to_validator_dict(m))
    proton_findings.extend(findings)

helium_findings = []
for m in helium_measurements:
    findings = validate_record(measurement_to_validator_dict(m))
    helium_findings.extend(findings)

print(f"Proton validation: {len(proton_findings)} findings across {len(proton_measurements)} records")
if proton_findings:
    for f in proton_findings[:5]:
        print(f"  {f}")
else:
    print("  All proton records passed validation.")

print(f"\nHelium validation: {len(helium_findings)} findings across {len(helium_measurements)} records")
if helium_findings:
    for f in helium_findings[:5]:
        print(f"  {f}")
else:
    print("  All helium records passed validation.")

## 6. Harmonise records and inspect uncertainty labels

The four-stage harmonisation pipeline applies:
1. **Species normalisation** — canonical species names
2. **Axis harmonisation** — common energy axis and unit
3. **Uncertainty labelling** — each error tagged as `published`, `derived`, or `assumed`
4. **Time-window normalisation** — UTC datetime fields

Since both sample CSVs provide `stat_err` and `sys_err_total` directly
from the publication table (no heuristic splitting or symmetrisation),
the `uncertainty_label` for diagonal mode is **`published`**.

Each output record carries a `provenance_json` field that merges the
original source metadata with harmonisation step information.

In [ ]:
parse_ctx = ParseContext(stat_err_from_table=True, sys_err_from_table=True)

proton_provenance = {
    "paper_doi": "10.1103/PhysRevLett.114.171103",
    "table_id": "T1",
    "file_url": "data/samples/proton_sample.csv",
    "source_type": "csv",
}

helium_provenance = {
    "paper_doi": "10.1103/PhysRevLett.115.211101",
    "table_id": "T1",
    "file_url": "data/samples/helium_sample.csv",
    "source_type": "csv",
}

proton_canonical = run_harmonisation_pipeline(
    proton_measurements, parse_ctx, proton_provenance
)
helium_canonical = run_harmonisation_pipeline(
    helium_measurements, parse_ctx, helium_provenance
)

print(f"{len(proton_canonical)} canonical proton records")
print(f"{len(helium_canonical)} canonical helium records")

# Display provenance_json explicitly
print("\n=== Proton provenance_json (first record) ===")
print(json.dumps(proton_canonical[0]["provenance_json"], indent=2))

print("\n=== Helium provenance_json (first record) ===")
print(json.dumps(helium_canonical[0]["provenance_json"], indent=2))

# Display uncertainty_label for diagonal mode
print(f"\nuncertainty_label for diagonal mode: {UNCERTAINTY_LABEL}")
print("Both stat and sys errors are read from the table, so the label is published.")

print("\nFirst canonical proton record:")
proton_canonical[0]

## 7. Build diagonal covariance matrices

Use published stat and sys errors to construct diagonal covariance
matrices for both species.  The `uncertainty_label` for diagonal mode
is `'published'` — no derived or assumed uncertainties are introduced.

In [ ]:
proton_stat = np.array([r["stat_err"] for r in proton_canonical])
proton_sys = np.array([r["sys_err_total"] for r in proton_canonical])
proton_cov = build_diagonal_covariance(proton_stat, proton_sys)

helium_stat = np.array([r["stat_err"] for r in helium_canonical])
helium_sys = np.array([r["sys_err_total"] for r in helium_canonical])
helium_cov = build_diagonal_covariance(helium_stat, helium_sys)

print(f"Proton covariance shape: {proton_cov.shape}")
print(f"Helium covariance shape: {helium_cov.shape}")
print(f"uncertainty_label: {UNCERTAINTY_LABEL}")
print(f"mode: {MODE}")
print(f"\nProton first 3 diagonal: {np.diag(proton_cov)[:3]}")
print(f"Helium first 3 diagonal: {np.diag(helium_cov)[:3]}")

## 7b. Chi-squared with a toy power-law model

With the diagonal covariance in hand we can compute a chi-squared
goodness-of-fit statistic.  Because the covariance is diagonal
(bins uncorrelated), the chi-squared simplifies to a sum over bins:

$$\chi^2 = \sum_i \frac{(y_i - f(x_i))^2}{\sigma_{\mathrm{stat},i}^2 + \sigma_{\mathrm{sys},i}^2}$$

We use a simple power-law model $f(x) = A \, x^{-\gamma}$ with
hand-picked normalisation $A$ and spectral index $\gamma$.  This is
**not** a fit — just a demonstration that the diagonal covariance
feeds correctly into a chi-squared calculation.

## 8. Assemble fit-ready datasets

Bundle arrays + metadata into fit-ready dicts for downstream consumers.

In [ ]:
proton_y = np.array([r["y_value"] for r in proton_canonical])
proton_x = np.array([r["x_centre"] for r in proton_canonical])

proton_fit = build_fit_dataset(
    y=proton_y, x=proton_x, covariance=proton_cov,
    uncertainty_label=UNCERTAINTY_LABEL,
    mode=MODE,
    provenance=proton_provenance,
    species="PROTON",
    x_axis_type="kinetic_energy_per_nucleon",
    y_unit="m-2 sr-1 s-1 GV-1",
)

helium_y = np.array([r["y_value"] for r in helium_canonical])
helium_x = np.array([r["x_centre"] for r in helium_canonical])

helium_fit = build_fit_dataset(
    y=helium_y, x=helium_x, covariance=helium_cov,
    uncertainty_label=UNCERTAINTY_LABEL,
    mode=MODE,
    provenance=helium_provenance,
    species="HELIUM",
    x_axis_type="kinetic_energy_per_nucleon",
    y_unit="m-2 sr-1 s-1 GV-1",
)

print(f"Proton fit dataset keys: {sorted(proton_fit.keys())}")
print(f"Proton n_points: {proton_fit['n_points']}")
print(f"Helium n_points: {helium_fit['n_points']}")

## 9. Export to parquet

Write both species to parquet files with provenance metadata embedded
as file-level parquet metadata.

In [ ]:
tmpdir = tempfile.mkdtemp()

proton_export_df = pd.DataFrame({
    "x_centre": proton_fit["x"],
    "y_value": proton_fit["y"],
    "stat_err": proton_stat,
    "sys_err_total": proton_sys,
})
proton_dataset = {
    "data": proton_export_df,
    "provenance": proton_provenance,
    "covariance_label": UNCERTAINTY_LABEL,
}
proton_parquet_path = Path(tmpdir) / "proton_fit.parquet"
proton_written = export_parquet(proton_dataset, proton_parquet_path)
print(f"Proton exported to: {proton_written}")

helium_export_df = pd.DataFrame({
    "x_centre": helium_fit["x"],
    "y_value": helium_fit["y"],
    "stat_err": helium_stat,
    "sys_err_total": helium_sys,
})
helium_dataset = {
    "data": helium_export_df,
    "provenance": helium_provenance,
    "covariance_label": UNCERTAINTY_LABEL,
}
helium_parquet_path = Path(tmpdir) / "helium_fit.parquet"
helium_written = export_parquet(helium_dataset, helium_parquet_path)
print(f"Helium exported to: {helium_written}")

## 10. Read back and verify

Read exported parquet files back and assert the row counts match the
original input records — a basic round-trip integrity check.

In [ ]:
proton_readback = pd.read_parquet(proton_written)
helium_readback = pd.read_parquet(helium_written)

print(f"Proton: read back {len(proton_readback)} rows from parquet")
print(f"Helium: read back {len(helium_readback)} rows from parquet")

assert len(proton_readback) == len(proton_records), (
    f"Proton row count mismatch: parquet has {len(proton_readback)}, "
    f"input had {len(proton_records)}"
)
assert len(helium_readback) == len(helium_records), (
    f"Helium row count mismatch: parquet has {len(helium_readback)}, "
    f"input had {len(helium_records)}"
)
print("Row count assertions passed.")
print("\nProton readback:")
proton_readback.head()